# US Macro Economic Observatory

This notebook builds an **interactive dashboard** for tracking the US economy using three key indicators:

| Indicator | Source | What It Measures |
|-----------|--------|------------------|
| **CPI (Consumer Price Index)** | Bureau of Labor Statistics | Inflation — how fast prices are rising |
| **HPI (House Price Index)** | FHFA | Housing market health — home value trends |
| **Unemployment Rate** | Bureau of Labor Statistics | Labor market strength |

All data comes from `SNOWFLAKE_PUBLIC_DATA_FREE` — no setup or data loading required.

**Flow:** Data Discovery → Data Extraction → Transformation → Interactive Dashboard → Insights

---
## Part 1: Data Discovery

Before building anything, let's explore what datasets are available and understand their structure.

In [ ]:
%%sql -r available_databases
-- What databases do we have access to?
SHOW DATABASES

In [ ]:
-- Preview the BLS Price dataset (CPI lives here)
-- Each row is a timeseries observation: a variable, a date, and a value
SELECT VARIABLE_NAME, GEO_ID, DATE, VALUE
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES
WHERE VARIABLE_NAME ILIKE '%CPI%All items%'
  AND GEO_ID = 'country/USA'
ORDER BY DATE DESC
LIMIT 10

In [ ]:
-- Preview the FHFA House Price Index
-- National-level purchase-only HPI, seasonally adjusted
SELECT VARIABLE_NAME, GEO_ID, DATE, VALUE
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FHFA_HOUSE_PRICE_TIMESERIES
WHERE VARIABLE_NAME ILIKE '%purchase-only%seasonally%'
  AND GEO_ID = 'country/USA'
ORDER BY DATE DESC
LIMIT 10

In [ ]:
-- Preview the BLS Employment dataset (unemployment rate)
-- State-level data that we'll aggregate to national average
SELECT VARIABLE_NAME, GEO_ID, DATE, VALUE
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES
WHERE VARIABLE_NAME ILIKE '%unemployment rate%seasonally adjusted%'
  AND DATE >= '2024-01-01'
ORDER BY DATE DESC
LIMIT 10

---
## Part 2: Data Extraction

Now we pull each indicator as a clean timeseries from 2010 onward. Each query:
- Filters to the exact variable we need
- Scopes to national-level data (or aggregates states)
- Orders by date for timeseries analysis

In [ ]:
%%sql -r cpi_data
-- CPI Index: monthly, national, not seasonally adjusted
-- This is the headline CPI number used to calculate inflation
SELECT DATE, VALUE AS CPI_INDEX
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES
WHERE VARIABLE_NAME = 'CPI: All items, Not seasonally adjusted, Monthly'
  AND GEO_ID = 'country/USA'
  AND DATE >= '2010-01-01'
ORDER BY DATE

In [ ]:
%%sql -r hpi_data
-- House Price Index: monthly, national, seasonally adjusted
-- Purchase-only index (excludes refinances) — better signal for real demand
SELECT DATE, VALUE AS HPI
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FHFA_HOUSE_PRICE_TIMESERIES
WHERE VARIABLE_NAME = 'FHFA_HPI traditional purchase-only monthly Seasonally Adjusted'
  AND GEO_ID = 'country/USA'
  AND DATE >= '2010-01-01'
ORDER BY DATE

In [ ]:
%%sql -r unemployment_data
-- Unemployment Rate: average across all states per month
-- We aggregate state-level data to get a national proxy
SELECT DATE, ROUND(AVG(VALUE), 2) AS UNEMPLOYMENT_RATE
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES
WHERE VARIABLE_NAME = 'Local Area Unemployment: Unemployment Rate, Seasonally adjusted, Monthly'
  AND DATE >= '2010-01-01'
GROUP BY DATE
ORDER BY DATE

---
## Part 3: Transformation

We merge all three indicators into a single panel dataset, then compute derived metrics:
- **YoY Inflation %** — 12-month percentage change in CPI
- **HPI YoY Growth %** — 12-month percentage change in house prices
- **Rolling Averages** — 6-month smoothed trends
- **Economic Regime** — classify each month as Goldilocks, Stagflation, Recession, or Overheating

In [ ]:
import pandas as pd
import numpy as np

# Convert SQL results to pandas
cpi_pdf = cpi_data.to_pandas() if not isinstance(cpi_data, pd.DataFrame) else cpi_data.copy()
hpi_pdf = hpi_data.to_pandas() if not isinstance(hpi_data, pd.DataFrame) else hpi_data.copy()
unemp_pdf = unemployment_data.to_pandas() if not isinstance(unemployment_data, pd.DataFrame) else unemployment_data.copy()

# Standardize date columns
for df in [cpi_pdf, hpi_pdf, unemp_pdf]:
    df['DATE'] = pd.to_datetime(df['DATE'])

# The unemployment rate is stored as a decimal (0.04 = 4%), convert to percentage
unemp_pdf['UNEMPLOYMENT_RATE'] = unemp_pdf['UNEMPLOYMENT_RATE'] * 100

# Merge all three on DATE
macro = cpi_pdf.merge(hpi_pdf, on='DATE', how='inner').merge(unemp_pdf, on='DATE', how='inner')

# Compute YoY inflation (12-month % change in CPI)
macro['YOY_INFLATION_PCT'] = macro['CPI_INDEX'].pct_change(12, fill_method=None) * 100

# Compute YoY HPI growth
macro['HPI_YOY_GROWTH_PCT'] = macro['HPI'].pct_change(12, fill_method=None) * 100

# Rolling 6-month averages for smoothing
macro['INFLATION_6M_AVG'] = macro['YOY_INFLATION_PCT'].rolling(6).mean()
macro['UNEMPLOYMENT_6M_AVG'] = macro['UNEMPLOYMENT_RATE'].rolling(6).mean()

# Economic regime classification
def classify_regime(row):
    infl = row['YOY_INFLATION_PCT']
    unemp = row['UNEMPLOYMENT_RATE']
    if pd.isna(infl) or pd.isna(unemp):
        return 'Unknown'
    if infl < 3 and unemp < 5:
        return 'Goldilocks'       # low inflation + low unemployment = ideal
    elif infl >= 3 and unemp >= 5:
        return 'Stagflation'      # high inflation + high unemployment = worst
    elif infl < 3 and unemp >= 5:
        return 'Recession Risk'   # low inflation + high unemployment
    else:
        return 'Overheating'      # high inflation + low unemployment

macro['REGIME'] = macro.apply(classify_regime, axis=1)
macro = macro.dropna(subset=['YOY_INFLATION_PCT']).reset_index(drop=True)

print(f"Macro panel: {len(macro)} months from {macro['DATE'].min().strftime('%Y-%m')} to {macro['DATE'].max().strftime('%Y-%m')}")
print(f"\nRegime distribution:")
print(macro['REGIME'].value_counts().to_string())
print(f"\nLatest values ({macro['DATE'].max().strftime('%b %Y')}):")
latest = macro.iloc[-1]
print(f"  Inflation:    {latest['YOY_INFLATION_PCT']:.1f}%")
print(f"  Unemployment: {latest['UNEMPLOYMENT_RATE']:.1f}%")
print(f"  HPI:          {latest['HPI']:.1f}")
print(f"  Regime:       {latest['REGIME']}")

---
## Part 4: Interactive Dashboard

Use the controls below to explore the data:
- **Date Range** — slide to zoom into specific periods
- **Metrics** — toggle which indicators to display
- **Regime Highlight** — select a regime to highlight on the chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output

# Prepare data for widgets
years = sorted(macro['DATE'].dt.year.unique())
min_year, max_year = int(years[0]), int(years[-1])

regime_colors = {
    'Goldilocks': '#2ecc71',
    'Stagflation': '#e74c3c',
    'Recession Risk': '#3498db',
    'Overheating': '#f39c12',
    'Unknown': '#95a5a6'
}

# Widgets
year_range = widgets.IntRangeSlider(
    value=[max_year - 5, max_year], min=min_year, max=max_year,
    step=1, description='Years:', continuous_update=False,
    layout=widgets.Layout(width='600px')
)

metric_checks = widgets.ToggleButtons(
    options=['All Metrics', 'Inflation Only', 'Unemployment Only', 'Housing Only'],
    value='All Metrics', description='Show:',
    layout=widgets.Layout(width='600px')
)

regime_dropdown = widgets.Dropdown(
    options=['None'] + list(regime_colors.keys()),
    value='None', description='Highlight:',
    layout=widgets.Layout(width='300px')
)

out = widgets.Output()

def update_dashboard(change=None):
    with out:
        clear_output(wait=True)
        
        # Filter by date range
        y_start, y_end = year_range.value
        mask = (macro['DATE'].dt.year >= y_start) & (macro['DATE'].dt.year <= y_end)
        df = macro[mask].copy()
        
        if len(df) == 0:
            print("No data in selected range.")
            return
        
        metric = metric_checks.value
        regime_hl = regime_dropdown.value
        
        # Determine which panels to show
        if metric == 'Inflation Only':
            panels = [('YOY_INFLATION_PCT', 'YoY Inflation %', '#e74c3c', 'INFLATION_6M_AVG')]
        elif metric == 'Unemployment Only':
            panels = [('UNEMPLOYMENT_RATE', 'Unemployment Rate %', '#3498db', 'UNEMPLOYMENT_6M_AVG')]
        elif metric == 'Housing Only':
            panels = [('HPI_YOY_GROWTH_PCT', 'HPI YoY Growth %', '#2ecc71', None)]
        else:
            panels = [
                ('YOY_INFLATION_PCT', 'YoY Inflation %', '#e74c3c', 'INFLATION_6M_AVG'),
                ('UNEMPLOYMENT_RATE', 'Unemployment Rate %', '#3498db', 'UNEMPLOYMENT_6M_AVG'),
                ('HPI_YOY_GROWTH_PCT', 'HPI YoY Growth %', '#2ecc71', None),
            ]
        
        n_panels = len(panels)
        fig, axes = plt.subplots(n_panels, 1, figsize=(14, 4 * n_panels), sharex=True)
        if n_panels == 1:
            axes = [axes]
        
        for ax, (col, label, color, avg_col) in zip(axes, panels):
            ax.plot(df['DATE'], df[col], color=color, linewidth=1.5, label=label)
            if avg_col and avg_col in df.columns:
                ax.plot(df['DATE'], df[avg_col], color=color, linewidth=2.5, alpha=0.4, linestyle='--', label='6M Avg')
            
            # Highlight selected regime
            if regime_hl != 'None':
                regime_mask = df['REGIME'] == regime_hl
                ax.fill_between(df['DATE'], df[col].min(), df[col].max(),
                                where=regime_mask, alpha=0.15,
                                color=regime_colors.get(regime_hl, '#95a5a6'),
                                label=regime_hl)
            
            ax.set_ylabel(label, fontsize=11)
            ax.legend(loc='upper left', fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='-')
        
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        axes[-1].xaxis.set_major_locator(mdates.YearLocator())
        plt.xticks(rotation=45)
        
        fig.suptitle(f'US Macro Observatory: {y_start}–{y_end}', fontsize=16, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.show()

# Connect widgets to update function
year_range.observe(update_dashboard, names='value')
metric_checks.observe(update_dashboard, names='value')
regime_dropdown.observe(update_dashboard, names='value')

# Layout and display
controls = widgets.VBox([
    widgets.HTML('<h3>Dashboard Controls</h3>'),
    year_range, metric_checks, regime_dropdown
])
display(widgets.VBox([controls, out]))
update_dashboard()

---
## Part 5: Correlation Analysis

How do these indicators move together? A **correlation matrix** reveals the relationships:
- Positive correlation → they move in the same direction
- Negative correlation → they move in opposite directions
- Near zero → no linear relationship

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute correlation matrix
corr_cols = ['YOY_INFLATION_PCT', 'UNEMPLOYMENT_RATE', 'HPI_YOY_GROWTH_PCT']
corr_labels = ['Inflation', 'Unemployment', 'HPI Growth']
corr_matrix = macro[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
for i in range(len(corr_labels)):
    for j in range(len(corr_labels)):
        axes[0].text(j, i, f'{corr_matrix.values[i, j]:.2f}',
                     ha='center', va='center', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(len(corr_labels)))
axes[0].set_yticks(range(len(corr_labels)))
axes[0].set_xticklabels(corr_labels, rotation=45)
axes[0].set_yticklabels(corr_labels)
axes[0].set_title('Correlation Matrix', fontsize=13, fontweight='bold')
fig.colorbar(im, ax=axes[0], shrink=0.8)

# Regime pie chart
regime_counts = macro['REGIME'].value_counts()
colors = [regime_colors.get(r, '#95a5a6') for r in regime_counts.index]
axes[1].pie(regime_counts.values, labels=regime_counts.index, colors=colors,
            autopct='%1.0f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Economic Regime Distribution (2010–Present)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey takeaways:")
print(f"  • Inflation vs Unemployment: {corr_matrix.loc['YOY_INFLATION_PCT', 'UNEMPLOYMENT_RATE']:.2f} (Phillips Curve relationship)")
print(f"  • Inflation vs HPI Growth:   {corr_matrix.loc['YOY_INFLATION_PCT', 'HPI_YOY_GROWTH_PCT']:.2f}")
print(f"  • Unemployment vs HPI Growth: {corr_matrix.loc['UNEMPLOYMENT_RATE', 'HPI_YOY_GROWTH_PCT']:.2f}")

---
## Summary

**What we built:**
1. Connected to Snowflake's free public datasets (BLS + FHFA)
2. Extracted CPI, House Price Index, and unemployment rate timeseries
3. Computed derived metrics: YoY inflation, HPI growth, rolling averages, economic regimes
4. Built an interactive dashboard with date range, metric, and regime filters
5. Analyzed correlations between the three major economic indicators

**Next:** See `02_tpch_retail_analytics.ipynb` for supply chain analysis, or `03_ai_data_assistant.ipynb` to chat with an AI that queries this data for you.